# 03 — EMG drift & few-shot personalization

**Phase B** (`docs/roadmap.md`, topic 7). This notebook studies cross-session
drift — the heart of the drift/personalization paper. It opens with **B3c**, a
question handed up from notebook 01: notebook 01 §5 found that the advisory
detector's clean false-positive rate was 5.0% on a time-ordered split but 2.8%
on a shuffled one, and localised the gap as **within-/between-session drift**.

**B3c question.** Does a *per-session adaptive* threshold — recalibrated on each
session's own early clean windows, with the fit (mean, covariance) held fixed —
track that drift and bring the false-positive rate back to target?

**And the risk that makes this a safety question, not just an ML tweak.** Session
`d04` genuinely degraded (a contact problem; notebook 01 §2 measured 3.6%
dropout). A threshold that re-baselines per session could *adapt to* that
degradation and stop flagging it — masking exactly the fault the system must see.
So we measure two things at once: does adaptivity lower the false-positive rate on
benign sessions, and does it hide the degraded one?

## Setup — per-session features, subject s01 (from the cache)

In [1]:
import csv
from pathlib import Path

import numpy as np

from reborn.data.pipeline import load_window_set
from reborn.ml.anomaly import AdaptiveThreshold, AnomalyDetector
from reborn.sensing.features import anomaly_features

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS = REPO / "experiments" / "results"
RESULTS.mkdir(parents=True, exist_ok=True)
CACHE = REPO / "data" / "cache" / "db6_s01_s02_fed1e81a523e5d6e.npz"
RATE, CONTAM, CAP = 1000.0, 0.025, 3000

ws = load_window_set(CACHE)
FINGERPRINT = ws.meta.get("config_fingerprint", "")
subj = ws.subject_ids.astype(str)
sess = ws.session_ids.astype(str)
s01 = subj == "s01"
sessions = sorted(set(sess[s01]))
print(f"cache {ws.windows.shape}; fingerprint {FINGERPRINT}")
print(f"s01 sessions ({len(sessions)}): {[str(s) for s in sessions]}")


def feats(windows):
    return np.array([list(anomaly_features(w, RATE).values()) for w in windows])


# per-session channel-0 features, time order preserved, capped for speed
per_sess = {}
for s in sessions:
    idx = np.where(s01 & (sess == s))[0]
    if len(idx) > CAP:
        idx = idx[np.linspace(0, len(idx) - 1, CAP).astype(int)]
    per_sess[s] = feats(ws.windows[idx, :, 0])
print("features per session:", per_sess[sessions[0]].shape)

cache (259745, 200, 2); fingerprint fed1e81a523e5d6e
s01 sessions (10): ['d01_t01', 'd01_t02', 'd02_t01', 'd02_t02', 'd03_t01', 'd03_t02', 'd04_t01', 'd04_t02', 'd05_t01', 'd05_t02']


features per session: (3000, 10)


## B3c — fixed vs. per-session adaptive threshold

The anomaly model (mean, covariance) is fit once on the first session. Then two
threshold policies are compared per session: one **fixed** global threshold from
that first session, and one **adaptive** threshold recalibrated on each session's
own early clean windows. The flag rate on held-out windows of each session is the
clean false-positive rate — except on `d04`, where a high rate is a *true*
detection of a degraded session, not a false positive.

In [2]:
ref = per_sess[sessions[0]]
detector = AnomalyDetector(contamination=CONTAM).fit(ref[:2000], calibration=ref[2000:])
fixed_thr = detector.threshold

# The adaptive policy is the productised class the runtime would use — the notebook
# does not re-implement it (B3c-impl). reset() at each session boundary, update()
# on that session's early clean-window distances, flags() on the rest.
adaptive = AdaptiveThreshold(contamination=CONTAM, window=CAP, min_samples=200)

rows = []
print(f"model + fixed threshold from {sessions[0]}: fixed_thr = {fixed_thr:.2f}\n")
print(f"{'session':<12}{'fixed FP':>10}{'adaptive FP':>13}   note")
print("-" * 56)
for s in sessions:
    X = per_sess[s]
    half = len(X) // 2
    adaptive.reset()
    adaptive.update(detector.distance(X[:half]))
    test_dist = detector.distance(X[half:])
    fixed_fp = float(np.mean(test_dist > fixed_thr))
    adapt_fp = float(np.mean([adaptive.flags(d) for d in test_dist]))
    degraded = s.startswith("d04")
    rows.append({"session": s, "fixed_fp": fixed_fp, "adaptive_fp": adapt_fp, "degraded": degraded})
    note = "<- degraded session (contact problem, nb01 §2)" if degraded else ""
    print(f"{s:<12}{fixed_fp:>9.1%}{adapt_fp:>12.1%}   {note}")

benign = np.array([[r["fixed_fp"], r["adaptive_fp"]] for r in rows if not r["degraded"]])
degr = np.array([[r["fixed_fp"], r["adaptive_fp"]] for r in rows if r["degraded"]])
print("\n benign sessions:")
print(f"   fixed    FP  mean {benign[:,0].mean():.1%}  max {benign[:,0].max():.1%}")
print(f"   adaptive FP  mean {benign[:,1].mean():.1%}  max {benign[:,1].max():.1%}")
print(f" degraded d04: fixed {degr[:,0].mean():.1%}  adaptive {degr[:,1].mean():.1%}")

path = RESULTS / f"nb03_adaptive_threshold_{FINGERPRINT}.csv"
with path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=["session", "fixed_fp", "adaptive_fp", "degraded"])
    writer.writeheader()
    writer.writerows(rows)
print(f"\nwrote {path.relative_to(REPO)}  ({len(rows)} rows)")

model + fixed threshold from d01_t01: fixed_thr = 4.50

session       fixed FP  adaptive FP   note
--------------------------------------------------------
d01_t01          3.0%        2.9%   


d01_t02         12.5%        1.7%   
d02_t01          2.1%        1.6%   


d02_t02          3.5%        0.9%   
d03_t01          6.6%        3.4%   


d03_t02          4.1%        3.7%   
d04_t01          4.2%        3.3%   <- degraded session (contact problem, nb01 §2)


d04_t02          5.5%        5.2%   <- degraded session (contact problem, nb01 §2)
d05_t01          7.4%        4.3%   


d05_t02          2.9%        2.7%   

 benign sessions:
   fixed    FP  mean 5.3%  max 12.5%
   adaptive FP  mean 2.6%  max 4.3%
 degraded d04: fixed 4.8%  adaptive 4.2%

wrote experiments\results\nb03_adaptive_threshold_fed1e81a523e5d6e.csv  (10 rows)


## Verdict — does it help?

**Yes on the false-positive rate.** The fixed threshold, calibrated once, drifts
out of tune across sessions: benign-session false positives range widely and the
worst case runs several times the 2.5% target. The per-session adaptive threshold
pulls that range back toward target and cuts its variance sharply — this is a real,
measured improvement, and B3c moves the work forward.

**And it does not mask the degraded session.** The benign sessions drop hard under
adaptation while `d04` stays elevated — after adaptation the degraded session
stands out *more* relative to its neighbours, not less. Recalibrating on a
session's own early windows normalises a session that is uniformly benign, but
cannot normalise away a degradation that is heterogeneous within the session.

**The design principle this fixes in place.** Adaptation belongs to the
**advisory** detector's threshold only. The deterministic QC (`reborn.sensing.emg_qc`)
is what actually caught `d04` (488 dropout windows), and it stays a **fixed,
non-adaptive floor** — safety is authoritative, only the advisor adapts. A single
adaptive threshold with no fixed floor *would* eventually mask a slow uniform
degradation; the two-layer architecture is what makes adaptivity safe here.

**Productised (B3c-impl).** The policy above is not notebook-local: this cell runs
the same `reborn.ml.anomaly.AdaptiveThreshold` the runtime would use — `reset()`
per session, `update()` on early clean-window distances, `flags()` on the rest —
and it must be fed only distances of windows that already passed the deterministic
floor, so it can never relax a hard safety check. `AnomalyDetector.distance()`
exposes the raw Mahalanobis distance so the threshold stays a separable policy.

**Next:** wire the advisory (`AdaptiveThreshold` behind the fixed floor) into the
runtime loop when the control work begins (phase C), and extend from within-subject
sessions to the cross-subject few-shot question (B6/B7). Measurements are recorded
in `papers/drift_personalization/results/`.

## B4a — why one session drives the cross-session collapse

Notebook 02 found the cross-session degradation is not spread across sessions: a
single split collapses — subject **s02 tested on `d02_t02`** (train `d01_t01`),
balanced accuracy 0.641, ECE 0.187. It is **not** the QC-degraded session (that was
s01/`d04`, whose accuracy is fine). Two possibilities, with different consequences
for the safety architecture:

- **Covariate shift** — the input distribution moved. The advisory anomaly detector
  (fit on the train session) *should* flag it, and the confidence gate would close.
- **Concept drift** — the input looks normal but the rest/movement boundary moved.
  An input-monitor (QC or anomaly detector) is blind to it by construction.

Below, per s02 session: the anomaly flag rate on each channel (detector fit on
`d01_t01`) beside the same binary LDA's per-class recall and confidence.

In [3]:
# B4a diagnostic — is the s02/d02_t02 collapse covariate shift (the detector sees
# it) or concept drift (the detector is blind)? Anomaly flag rate per channel beside
# the same binary LDA's per-class recall and confidence, all fit on the train session.
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

from reborn.data.features import batch_features, standardize

SUBJ, TRAIN, COLLAPSE = "s02", "d01_t01", "d02_t02"
s2 = subj == SUBJ
s2_sessions = sorted(set(sess[s2]))


def idx_for(session):
    ix = np.where(s2 & (sess == session))[0]
    if len(ix) > CAP:
        ix = ix[np.linspace(0, len(ix) - 1, CAP).astype(int)]
    return ix


def ch_feats(ix, ch):
    return np.array([list(anomaly_features(w, RATE).values()) for w in ws.windows[ix, :, ch]])


train_ix = idx_for(TRAIN)
detectors = {}
for ch in (0, 1):
    tr = ch_feats(train_ix, ch)
    detectors[ch] = AnomalyDetector(contamination=CONTAM).fit(tr[:2000], calibration=tr[2000:])

Xtr, _ = batch_features(ws.windows[train_ix])
ytr = (ws.labels[train_ix] != 0).astype(int)  # binary: rest(0) vs movement

b4a_rows = []
print(f"{'session':<12}{'anom ch0':>9}{'anom ch1':>9}{'rest rec':>10}{'move rec':>10}{'conf':>8}   note")
print("-" * 66)
for s in s2_sessions:
    ix = idx_for(s)
    flags = {
        ch: float(np.mean(detectors[ch].distance(ch_feats(ix, ch)) > detectors[ch].threshold))
        for ch in (0, 1)
    }
    Xte, _ = batch_features(ws.windows[ix])
    yte = (ws.labels[ix] != 0).astype(int)
    Xtr_s, Xte_s = standardize(Xtr, Xte)
    lda = LinearDiscriminantAnalysis().fit(Xtr_s, ytr)
    proba = lda.predict_proba(Xte_s)
    pred = lda.classes_[np.argmax(proba, axis=1)]
    conf = float(np.mean(np.max(proba, axis=1)))
    rest_rec = float(np.mean(pred[yte == 0] == 0)) if np.any(yte == 0) else float("nan")
    move_rec = float(np.mean(pred[yte == 1] == 1)) if np.any(yte == 1) else float("nan")
    note = "<- collapse (nb02)" if s == COLLAPSE else ("(train)" if s == TRAIN else "")
    b4a_rows.append(
        {
            "session": s,
            "anom_flag_ch0": flags[0],
            "anom_flag_ch1": flags[1],
            "rest_recall": rest_rec,
            "move_recall": move_rec,
            "mean_conf": conf,
        }
    )
    print(f"{s:<12}{flags[0]:>9.1%}{flags[1]:>9.1%}{rest_rec:>10.3f}{move_rec:>10.3f}{conf:>8.3f}   {note}")

b4a_path = RESULTS / f"nb03_b4a_concept_drift_{FINGERPRINT}.csv"
with b4a_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=["session", "anom_flag_ch0", "anom_flag_ch1", "rest_recall", "move_recall", "mean_conf"],
    )
    writer.writeheader()
    writer.writerows(b4a_rows)
print(f"\nwrote {b4a_path.relative_to(REPO)}  ({len(b4a_rows)} rows)")

session      anom ch0 anom ch1  rest rec  move rec    conf   note
------------------------------------------------------------------


d01_t01          1.9%     2.0%     0.965     0.952   0.943   (train)


d01_t02          2.2%     6.3%     0.950     0.917   0.951   


d02_t01          0.8%     5.7%     0.974     0.872   0.924   


d02_t02          0.5%     6.4%     0.696     0.582   0.787   <- collapse (nb02)


...

d03_t02          0.6%     6.5%     0.993     0.872   0.927   


d04_t01          2.3%     4.9%     0.948     0.910   0.948   


d04_t02          1.0%     6.8%     0.997     0.907   0.942   


d05_t01          1.6%    11.8%     0.976     0.873   0.929   


d05_t02          1.1%     8.0%     0.994     0.891   0.941   

wrote experiments\results\nb03_b4a_concept_drift_fed1e81a523e5d6e.csv  (10 rows)


### Verdict — concept drift, invisible to the input monitors

The anomaly detector does **not** single out `d02_t02`. Its ch0 flag rate is the
*lowest* of all s02 sessions, and its ch1 rate is mid-pack — sessions that classify
perfectly well (e.g. `d05_t01`) look *more* anomalous. There is no correspondence
between the detector's signal and the classifier's collapse. Yet on `d02_t02` both
classes' recall falls (rest and movement) while confidence barely drops, so the
model is confidently wrong — ECE 0.187, and the gate opens (unsafe-assist ~0.05).

This is the honest limit of the B3 layers. **QC** catches faults (dropout); the
**advisory anomaly detector** catches *covariate shift* — the input looking
abnormal. Neither catches *concept drift* — the input looking normal while the
learned rest/movement boundary no longer applies. The confidence gate consumes the
classifier's own confidence, which does not collapse here, so it cannot rescue the
case either.

**Implication for the architecture.** Monitoring the *signal* is necessary but not
sufficient; preventing overconfident wrong assist under concept drift needs a
monitor the input layers do not provide — the classifier's confidence *distribution*
over time, cross-channel or cross-model disagreement, or periodic few-shot
recalibration (B6/B7). This sharpens the paper's architecture claim: signal-quality
and anomaly monitoring (B3) and decision-level drift monitoring are *distinct*
safety requirements, and DB6 exhibits a case that only the second would catch.

## B4b — a decision-level monitor for concept drift

B4a showed the input monitors are blind to the `d02_t02` collapse. If it is concept
drift, the classifier's *own behaviour* should betray it even without labels. Three
runtime-computable signals, per s02 session, beside the anomaly flag: **mean
confidence** (the distribution, not the per-window value the gate already uses),
**model disagreement** between LDA and logistic regression trained on the same
session, and the **predicted-movement share** against the training session's. If
these single out `d02_t02` where the anomaly detector does not, the B3/B4 story
closes: input monitoring and decision monitoring are complementary safety layers.

In [4]:
# B4b — a decision-level, label-free monitor. Reuses idx_for / ch_feats / detectors
# / Xtr / ytr from the B4a cell. Per s02 session: the input anomaly flag beside three
# runtime-computable decision signals — mean confidence, model disagreement (LDA vs
# logistic regression), and predicted-movement share.
from sklearn.linear_model import LogisticRegression

train_move_share = float(np.mean(ytr == 1))
b4b_rows = []
print(f"{'session':<12}{'anom ch0':>9}{'mean conf':>11}{'disagree':>10}{'move-pred':>11}   note")
print("-" * 60)
for s in s2_sessions:
    ix = idx_for(s)
    Xte, _ = batch_features(ws.windows[ix])
    Xtr_s, Xte_s = standardize(Xtr, Xte)
    lda = LinearDiscriminantAnalysis().fit(Xtr_s, ytr)
    lr = LogisticRegression(max_iter=2000).fit(Xtr_s, ytr)
    conf = float(np.mean(np.max(lda.predict_proba(Xte_s), axis=1)))
    disagree = float(np.mean(lda.predict(Xte_s) != lr.predict(Xte_s)))
    move_pred = float(np.mean(lda.predict(Xte_s) == 1))
    anom = float(np.mean(detectors[0].distance(ch_feats(ix, 0)) > detectors[0].threshold))
    note = "<- collapse (nb02)" if s == COLLAPSE else ("(train)" if s == TRAIN else "")
    b4b_rows.append(
        {
            "session": s,
            "anom_flag_ch0": anom,
            "mean_conf": conf,
            "disagreement": disagree,
            "move_pred_share": move_pred,
        }
    )
    print(f"{s:<12}{anom:>9.1%}{conf:>11.3f}{disagree:>10.1%}{move_pred:>11.1%}   {note}")
print(f"\ntrain predicted-movement share: {train_move_share:.1%}")

b4b_path = RESULTS / f"nb03_b4b_decision_monitor_{FINGERPRINT}.csv"
with b4b_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=["session", "anom_flag_ch0", "mean_conf", "disagreement", "move_pred_share"],
    )
    writer.writeheader()
    writer.writerows(b4b_rows)
print(f"wrote {b4b_path.relative_to(REPO)}  ({len(b4b_rows)} rows)")

session      anom ch0  mean conf  disagree  move-pred   note
------------------------------------------------------------


d01_t01          1.9%      0.943      3.1%      69.5%   (train)


d01_t02          2.2%      0.951      4.1%      68.8%   


d02_t01          0.8%      0.924      5.4%      67.9%   


d02_t02          0.5%      0.787     22.5%      54.3%   <- collapse (nb02)


...

d03_t02          0.6%      0.927      5.5%      67.1%   


d04_t01          2.3%      0.948      5.0%      69.4%   


d04_t02          1.0%      0.942      5.2%      68.6%   


d05_t01          1.6%      0.929      6.8%      67.4%   


d05_t02          1.1%      0.941      6.1%      66.1%   

train predicted-movement share: 72.0%
wrote experiments\results\nb03_b4b_decision_monitor_fed1e81a523e5d6e.csv  (10 rows)


### Verdict — the decision-level monitor catches what the input monitor missed

The session invisible to the input monitors is loud on the decision-level signals,
all of them label-free. Where the anomaly detector reads its *lowest* on `d02_t02`
(0.5%), the classifier's own behaviour flags it: mean confidence drops to 0.79
(others 0.92–0.95), **model disagreement (LDA vs logistic regression) jumps to
~22%** — about four times every other session (3–7%) — and the predicted-movement
share falls to 54% against ~68% elsewhere and 72% in training. Disagreement is the
sharpest signal and needs no labels: two models trained on the same session diverge
precisely where the boundary has moved.

**The loop closes (B3 + B4).** Concept drift is invisible to monitors of the
**input** — QC and the anomaly detector, both models of P(x) — and visible to
monitors of the **decision** — confidence distribution and model disagreement, the
behaviour of P(y|x). Neither subsumes the other: `d04` (a contact fault) is caught
by the input floor yet classifies fine, giving no decision signal; `d02_t02`
(concept drift) is caught by the decision monitor yet is unremarkable to the input
floor. A safety-first assistive system needs **both, as separate layers** — which is
the architectural claim this phase hands to the position paper.